# Croatian 5-Letter Word Ladder — Distance Regression Training Data

Generate training examples for fine-tuning BERT to predict **BFS shortest-path distance** between word pairs in the Croatian 5-letter word-ladder graph.

- **Full vocab**: `data/islands/croatian_5_largest_island.txt` (largest connected component, 11,052 words)
- **Alphabet**: `abcdefghijklmnopqrstuvwxyzčćđšž` (31 Croatian letters)
- **Format**: regression — input `(word_a, word_b)`, label = shortest-path distance (integer)
- **Output**: `data/training/wordladder_croatian5_{train,val,test}.csv`
- **Split**: by word pair; stratified by distance bin
- **Defaults**: ~**600k** examples, **4000** BFS sources, **weighted sampling**

This notebook mirrors `05_english_5_letter_training.ipynb` but targets the **Croatian 5-letter** graph.

In [ ]:
import random
from pathlib import Path

import networkx as nx
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

SEED = 42
random.seed(SEED)

BASE = Path("../data")
ISLANDS = BASE / "islands"
TRAINING = BASE / "training"
TRAINING.mkdir(parents=True, exist_ok=True)

FULL_VOCAB = ISLANDS / "croatian_5_largest_island.txt"
CROATIAN_LETTERS = "abcdefghijklmnopqrstuvwxyz\u010d\u0107\u0111\u0161\u017e"


def load_words(path: Path) -> set[str]:
    return {w.strip().lower() for w in path.read_text(encoding="utf-8").splitlines() if w.strip()}


def one_letter_neighbors(w: str, vocab: set[str]) -> set[str]:
    neighbors = set()
    for i in range(len(w)):
        for c in CROATIAN_LETTERS:
            if c != w[i]:
                cand = w[:i] + c + w[i + 1:]
                if cand in vocab:
                    neighbors.add(cand)
    return neighbors


vocab = load_words(FULL_VOCAB)
print(f"Full vocab: {FULL_VOCAB.name} ({len(vocab)} words)")
print(f"Alphabet: {CROATIAN_LETTERS} ({len(CROATIAN_LETTERS)} letters)")

In [ ]:
# 2. Build graph from full vocab
G = nx.Graph()
G.add_nodes_from(vocab)
for w in tqdm(vocab, desc="Building graph"):
    for nb in one_letter_neighbors(w, vocab):
        if w < nb:
            G.add_edge(w, nb)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

In [ ]:
# 3. Compute BFS distances from sampled source words
NUM_SOURCES = 4000
vocab_list = sorted(vocab)
sources = random.sample(vocab_list, min(NUM_SOURCES, len(vocab_list)))

raw_pairs = []
for src in tqdm(sources, desc="BFS from source words"):
    lengths = nx.single_source_shortest_path_length(G, src)
    for tgt, dist in lengths.items():
        if tgt != src:
            raw_pairs.append((src, tgt, dist))

# Deduplicate symmetric pairs: keep only one of (a,b) and (b,a)
seen = set()
unique_pairs = []
for a, b, d in raw_pairs:
    key = (min(a, b), max(a, b))
    if key not in seen:
        seen.add(key)
        unique_pairs.append((a, b, d))
raw_pairs = unique_pairs

print(f"Unique pairs from {len(sources)} BFS sources: {len(raw_pairs)}")

In [ ]:
# 4. Subsample with weighted distance coverage (oversample short distances)
from collections import Counter, defaultdict

TARGET_EXAMPLES = 600_000

by_dist = defaultdict(list)
for item in raw_pairs:
    by_dist[item[2]].append(item)

dist_values = sorted(by_dist.keys())


def dist_weight(d: int) -> float:
    """Higher weight → more training rows at that graph distance.
    Short distances matter most for A* (picking the best 1-step neighbor)."""
    if d <= 3:
        return 3.0
    if d <= 6:
        return 2.0
    if d <= 9:
        return 1.5
    return 1.0


weights = {d: dist_weight(d) for d in dist_values}
wsum = sum(weights[d] for d in dist_values)
# Largest-remainder method so row counts sum exactly to TARGET_EXAMPLES
targets_float = {d: TARGET_EXAMPLES * weights[d] / wsum for d in dist_values}
alloc = {d: int(targets_float[d]) for d in dist_values}
shortfall = TARGET_EXAMPLES - sum(alloc.values())
frac_order = sorted(dist_values, key=lambda d: -(targets_float[d] % 1.0))
i = 0
while shortfall > 0:
    alloc[frac_order[i % len(frac_order)]] += 1
    shortfall -= 1
    i += 1

sampled = []
for d in dist_values:
    pool = by_dist[d]
    n = min(len(pool), alloc[d])
    sampled.extend(random.sample(pool, n))

if len(sampled) < TARGET_EXAMPLES:
    used = {(a, b) for a, b, _ in sampled}
    extra = [(a, b, d) for a, b, d in raw_pairs if (a, b) not in used]
    random.shuffle(extra)
    sampled.extend(extra[: TARGET_EXAMPLES - len(sampled)])

random.shuffle(sampled)
sampled = sampled[:TARGET_EXAMPLES]

# Randomly swap word order (distance is symmetric; model should learn both directions)
examples = []
for a, b, d in sampled:
    if random.random() < 0.5:
        a, b = b, a
    examples.append({"text_a": a, "text_b": b, "label": d})

print(f"Final examples: {len(examples)}")

dist_counts = Counter(ex["label"] for ex in examples)
print("\nDistance distribution:")
for d in sorted(dist_counts):
    print(f"  d={d}: {dist_counts[d]:>6} ({dist_counts[d] / len(examples):.1%})")

In [ ]:
# 5. Split by pair — stratified by distance bin (short 1–3, medium 4–6, long 7+)

pair_to_dist = {(ex["text_a"], ex["text_b"]): ex["label"] for ex in examples}
pairs_list = list(pair_to_dist.keys())
dist_bins = [0 if pair_to_dist[p] <= 3 else (1 if pair_to_dist[p] <= 6 else 2) for p in pairs_list]

bins_to_pairs = defaultdict(list)
for p, b in zip(pairs_list, dist_bins):
    bins_to_pairs[b].append(p)

train_set, val_set, test_set = set(), set(), set()
for bin_id, bin_pairs in bins_to_pairs.items():
    random.shuffle(bin_pairs)
    n = len(bin_pairs)
    t1 = int(0.90 * n)
    t2 = int(0.95 * n)
    train_set.update(bin_pairs[:t1])
    val_set.update(bin_pairs[t1:t2])
    test_set.update(bin_pairs[t2:])

train_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in train_set]
val_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in val_set]
test_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in test_set]

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    pd.DataFrame(data).to_csv(TRAINING / f"wordladder_croatian5_{name}.csv", index=False)

print(f"\n=== Final stats ===")
print(f"Total: {len(examples)}")
print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")
print(f"Label range: {min(ex['label'] for ex in examples)}–{max(ex['label'] for ex in examples)}")
print(f"Mean distance: {sum(ex['label'] for ex in examples) / len(examples):.2f}")
print(f"Saved to {TRAINING}/")

In [ ]:
# Done — CSVs ready for notebook 10 (BERT distance regression fine-tuning for Croatian 5-letter).
print("Data generation complete. Run notebook 10 to fine-tune.")